# Eurex GraphQL API Tutorial

[![Open In Colab](https://img.shields.io/badge/Google%20Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=white)](https://colab.research.google.com/github/ViktorHD/eurex-api/blob/main/notebooks/eurex_graphql_tutorial.ipynb) 
[![Binder](https://img.shields.io/badge/Binder-579ACA?style=for-the-badge&logo=binder&logoColor=white)](https://mybinder.org/v2/gh/ViktorHD/eurex-api/main?labpath=notebooks%2Feurex_graphql_tutorial.ipynb)

This notebook provides a complete guide for Python users to interact with the [Eurex GraphQL API](https://www.eurex.com/ex-en/data/free-reference-data-api).

## 1. Setup & Authentication

To access the API, you need an API key from the Deutsche Börse Digital Business Platform. For this tutorial, we use the public demo key available on the official Eurex API page.

In [ ]:
import sys
!{sys.executable} -m pip install pandas
import requests
import json
import pandas as pd
from datetime import datetime, timedelta

API_URL = "https://api.developer.deutsche-boerse.com/eurex-prod-graphql/"
# Public demo key from: https://www.eurex.com/ex-en/data/free-reference-data-api
API_KEY = "68cdafd2-c5c1-49be-8558-37244ab4f513"

headers = {
    "Content-Type": "application/json",
    "X-DBP-APIKEY": API_KEY
}

## 2. Basic Querying

The most basic way to query the API is to use the `requests` library to send a POST request with a GraphQL query string.

In [ ]:
def run_query(query):
    payload = {'query': query}
    response = requests.post(API_URL, json=payload, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Query failed with status code {response.status_code}: {response.text}")

# Simple query for products
query = """
query {
  ProductInfos {
    data {
      Product
      Name
      ProductType
    }
  }
}
"""

result = run_query(query)
products_df = pd.DataFrame(result['data']['ProductInfos']['data'])
print(f"Found {len(products_df)} products")
products_df.head()

## 3. Advanced Filtering

You can use the `filter` argument to narrow down your results. Filters support operators like `eq` (equals), `beginsWith`, `contains`, `gt` (greater than), `lt` (less than), and `between` (takes a list of two values).

In [ ]:
# Filter for Index Futures that start with 'FD'
advanced_filter_query = """
query {
  ProductInfos(filter: { 
    Product: { beginsWith: \"FD\" }, 
    ProductType: { eq: \"INDEX FUTURES\" } 
  }) {
    data {
      Product
      Name
      ProductType
      Currency
    }
  }
}
"""

result = run_query(advanced_filter_query)
filtered_products_df = pd.DataFrame(result['data']['ProductInfos']['data'])
filtered_products_df

## 4. Querying Different Entities

The API provides various entities beyond products and contracts, such as `SettlementPrices`, `Holidays`, and `TickRules`.

In [ ]:
# Query Settlement Prices and Holidays for a product
multi_entity_query = """
query {
  SettlementPrices(filter: { Product: { eq: \"FDAX\" } }) {
    data {
      SettlementDate
      SettlementPrice
      ContractID
    }
  }
  Holidays(filter: { Product: { eq: \"FDAX\" } }) {
    data {
      Holiday
    }
  }
}
"""

result = run_query(multi_entity_query)
prices_df = pd.DataFrame(result['data']['SettlementPrices']['data'])
holidays_df = pd.DataFrame(result['data']['Holidays']['data'])

print("Latest Settlement Prices:")
display(prices_df.head())
print("\nProduct Holidays:")
display(holidays_df.head())

## 5. Real-world Scenario: Upcoming Expirations

Find all contracts for 'FDAX' expiring in the next 90 days using the `between` filter.

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")
next_quarter = (datetime.now() + timedelta(days=90)).strftime("%Y-%m-%d")

expirations_query = f"""
query {{
  Contracts(filter: {{ 
    Product: {{ eq: \"FDAX\" }}, 
    ExpirationDate: {{ between: [\"{today}\", \"{next_quarter}\"] }}
  }}) {{
    data {{
      Contract
      ExpirationDate
      ContractID
    }}
  }}
}}
"""

result = run_query(expirations_query)
expiring_df = pd.DataFrame(result['data']['Contracts']['data'])
print(f"Found {len(expiring_df)} contracts expiring between {today} and {next_quarter}")
expiring_df

## 6. Data Merging with Pandas

Often you need to combine info from multiple entities. Here we join `ProductInfos` with `Contracts`.

In [ ]:
# 1. Get info for a set of products
prods_query = """
query {
  ProductInfos(filter: { Product: { beginsWith: \"OES\" } }) {
    data {
      Product
      Name
      UnderlyingName
    }
  }
}
"""
prods_data = run_query(prods_query)['data']['ProductInfos']['data']
df_prods = pd.DataFrame(prods_data)

# 2. Get contracts for those products
all_contracts = []
for p in prods_data:
    product_code = p['Product']
    contract_query = f"""
    query {{
      Contracts(filter: {{ Product: {{ eq: \"{product_code}\" }} }}) {{
        data {{
          Product
          Contract
          ExpirationDate
          ISIN
        }}
      }}
    }}
    """
    res = run_query(contract_query)
    all_contracts.extend(res['data']['Contracts']['data'])

df_contracts = pd.DataFrame(all_contracts)

# 3. Merge
merged_df = pd.merge(df_contracts, df_prods, on="Product")
merged_df.head()

## 7. Visualizing Data: Timeline of Front Month Contract

Instead of looking at the term structure, it's often useful to analyze the price history of a single contract over time. Here we take the front month FDAX contract (the closest to expiration) from our earlier query and plot its historical settlement prices as a timeline.

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib
import matplotlib.pyplot as plt

if not expiring_df.empty:
    # Get the front month contract name and integer ID
    sorted_expiring = expiring_df.sort_values('ExpirationDate')
    front_month_contract_name = sorted_expiring.iloc[0]['Contract']
    front_month_contract_id = int(sorted_expiring.iloc[0]['ContractID'])
    print(f"Plotting timeline for front month contract: {front_month_contract_name} (ID: {front_month_contract_id})")
    
    # Query historical settlement prices for this specific product and contract ID
    history_query = f"""
    query {{
      SettlementPrices(filter: {{ 
        Product: {{ eq: \"FDAX\" }},
        ContractID: {{ eq: {front_month_contract_id} }}
      }}) {{
        data {{
          SettlementDate
          SettlementPrice
        }}
      }}
    }}
    """
    
    hist_result = run_query(history_query)
    hist_df = pd.DataFrame(hist_result['data']['SettlementPrices']['data'])
    
    if not hist_df.empty:
        # Convert to datetime and sort
        hist_df['Date'] = pd.to_datetime(hist_df['SettlementDate'])
        hist_df = hist_df.sort_values('Date')
        
        plt.figure(figsize=(10, 6))
        plt.plot(hist_df['Date'], hist_df['SettlementPrice'], marker='', linestyle='-', color='#1f77b4', linewidth=2)
        plt.title(f"{front_month_contract_name} Historical Settlement Prices")
        plt.xlabel("Date")
        plt.ylabel("Settlement Price")
        plt.xticks(rotation=45)
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
    else:
        print(f"No historical settlement prices found for {front_month_contract_name}.")
else:
    print("No front month contract available to plot.")

## 8. Schema Introspection

GraphQL is self-documenting. You can use introspection queries to discover the schema programmatically, including all available queries, types, and their fields. This is useful when you want to know what data you can request without checking external documentation.

In [ ]:
introspection_query = """
query {
  __schema {
    queryType {
      fields {
        name
      }
    }
  }
}
"""

intro_res = run_query(introspection_query)
queries = intro_res["data"]["__schema"]["queryType"]["fields"]

print("Available Queries in Eurex API:")
for q in queries:
    print(f"- {q['name']}")

## 9. Exporting Data

Once you have retrieved and structured your data, you might want to save it locally for use in Excel or other tools. Pandas makes it easy to export your DataFrames to CSV.

In [ ]:
# Export the merged dataframe from the previous step to a CSV file
merged_df.to_csv("eurex_contracts.csv", index=False)
print("Data successfully exported to eurex_contracts.csv")
# You can also use .to_excel() if you have openpyxl installed